# Completion Basics

---

## Use case

La **completion** est la primitive fondamentale des LLMs : envoyer un prompt, recevoir une réponse.

Ce notebook couvre les mécaniques essentielles :
- Client synchrone et asynchrone
- Paramètres clés (temperature, max_tokens)
- Streaming
- Limites fondamentales du modèle (mémoire, knowledge cutoff)
- Transition vers LiteLLM pour une approche provider-agnostic

## Stack

- **OpenAI SDK** — client Python officiel, utilisé ici pour la pédagogie
- **python-dotenv** — chargement des variables d'environnement depuis `.env`
- **LiteLLM** — abstraction provider-agnostic, introduite en fin de notebook

## Setup

**En local**
1. Copier `.env.example` en `.env` à la racine du repo
2. Renseigner `OPENAI_API_KEY`
3. Lancer le notebook avec ton environnement uv ou jupyter habituel

**Sur Google Colab**
1. Ouvrir les Secrets (icône 🔑 dans le panneau gauche)
2. Ajouter un secret `OPENAI_API_KEY` avec ta clé
3. Activer l'accès au secret pour ce notebook

In [ ]:
%pip install openai python-dotenv litellm -q

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
except ImportError:
    pass  # Colab — variables disponibles via les secrets

assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY manquante — voir .env.example (local) ou Secrets (Colab)"
)

## Client synchrone

L'API OpenAI s'articule autour de deux concepts simples :
- **system prompt** (`instructions`):  le comportement global du modèle
- **user prompt** (`input`): le message de l'utilisateur

La réponse contient la completion mais aussi des métadonnées utiles : usage (tokens consommés), model utilisé, etc.

In [ ]:
from openai import OpenAI

# API key chargée automatiquement depuis OPENAI_API_KEY
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-mini",
    instructions="You are a helpful assistant.",
    input="What is a Python tuple? Answer in one sentence.",
)

# Texte de la réponse
print(response.output_text)

In [ ]:
# Tokens consommés — utile pour estimer les coûts
print("Usage:", response.usage)

# Output complet — contient le role, le contenu, etc.
print("Output:", response.output)

# Model effectivement utilisé
print("Model:", response.model)

## Client asynchrone

Le client asynchrone (`AsyncOpenAI`) est utile dès qu'on veut :
- Faire plusieurs appels en parallèle (`asyncio.gather`)
- Intégrer dans une application async (FastAPI, etc.)
- Combiner completion et streaming sans bloquer le thread principal

Dans un notebook Jupyter, `await` est utilisable directement, pas besoin de `asyncio.run()`.

In [ ]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI()

response = await async_client.responses.create(
    model="gpt-4.1-mini",
    instructions="You are a helpful assistant.",
    input="What is a Python tuple? Answer in one sentence.",
)

print(response.output_text)

> **Note** — `instructions` + `input` est un raccourci pratique. Le format explicite avec une liste de messages est équivalent et plus courant dans les intégrations :

In [ ]:
# Format équivalent — plus courant dans les intégrations
response = await async_client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is a Python tuple? Answer in one sentence."},
    ],
)

print(response.output_text)

## Paramètres clés

Trois paramètres influencent directement la qualité et le coût des completions :

- **`temperature`** — créativité de la réponse. `0` = déterministe, `1` = plus créatif. Par défaut `1`.
- **`max_output_tokens`** — nombre maximum de tokens générés. Utile pour maîtriser les coûts et forcer la concision.
- **`top_p`** — alternative à `temperature` pour contrôler la diversité. En pratique, on ajuste l'un ou l'autre, pas les deux.

> **Règle empirique** — tâches factuelles/code → `temperature` basse (0–0.3). Génération créative → `temperature` haute (0.7–1).

### Temperature

In [ ]:
from openai import OpenAI

client = OpenAI()

prompt = "Give me a one-sentence tagline for a coffee shop."

for temperature in [0, 0.7, 1.5]:
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt,
        temperature=temperature,
    )
    print(f"temperature={temperature} → {response.output_text}")

### Max tokens

In [ ]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Explain how the internet works.",
    max_output_tokens=50,
)

print(response.output_text)
print(f"\nTokens utilisés — prompt: {response.usage.input_tokens}, completion: {response.usage.output_tokens}")

## Streaming

Par défaut, la réponse est renvoyée en une seule fois une fois la génération terminée.
Le streaming envoie les tokens au fur et à mesure — ce qui change deux choses :

- **UX** — l'utilisateur voit la réponse apparaître progressivement, sans attente perçue
- **TTFB** (Time To First Byte) — le premier token arrive bien avant la fin de la génération

Indispensable en production pour toute interface conversationnelle.

In [ ]:
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from IPython.display import display

async_client = AsyncOpenAI()

# display_id permet de mettre à jour la même cellule output plutôt que d'appendre
handle = display("", display_id=True)

stream = await async_client.responses.create(
    model="gpt-4.1-mini",
    input="Explain how the internet works in a few sentences.",
    stream=True,
)

resp = ""
async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        resp += event.delta
        handle.update(resp)

## Limites fondamentales

Deux limites importantes à comprendre avant d'aller plus loin :

**Pas de mémoire** — chaque appel est indépendant. Le modèle ne se souvient pas des échanges précédents. C'est au développeur de gérer le contexte conversationnel en passant l'historique explicitement à chaque appel. → `02-augmentation/memoire`

**Knowledge cutoff** — le modèle ne connaît que les données sur lesquelles il a été entraîné. Il ne sait rien des événements récents, et rien de vos données internes. → `02-augmentation/rag`

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Hi! My name is Laurent.",
)
print("Response 1:", response.output_text)

# Nouvel appel — le modèle n'a aucun souvenir de l'échange précédent
response_2 = client.responses.create(
    model="gpt-4.1-mini",
    input="What is my name?",
)
print("Response 2:", response_2.output_text)

In [ ]:
# Événement récent — le modèle ne sait pas
response = client.responses.create(
    model="gpt-4.1-mini",
    input="How many medals did France win at the 2026 Winter Olympics?",
)
print("Événement récent:", response.output_text)

# Donnée interne — le modèle ne sait pas non plus
response_2 = client.responses.create(
    model="gpt-4.1-mini",
    input="How do I request time off at Younup?",
)
print("Donnée interne:", response_2.output_text)

## Vers une approche provider-agnostic avec LiteLLM

Jusqu'ici on a utilisé le SDK OpenAI directement — idéal pour comprendre la mécanique.

En pratique, on veut souvent pouvoir switcher de provider sans réécrire le code : changer de modèle selon les coûts, tester Anthropic vs OpenAI, utiliser un modèle local via Ollama, etc.

**LiteLLM** expose une interface unifiée compatible avec ~100 providers. Le format est identique à l'API OpenAI — seul le nom du modèle change.

Les providers sont identifiés par un préfixe :
- `openai/gpt-4.1-mini`
- `anthropic/claude-haiku-4-5`
- `ollama/llama3`

C'est le pattern qu'on utilisera dans tous les notebooks suivants.

In [ ]:
# Changer ici pour switcher de provider
MODEL = "openai/gpt-4.1-mini"
# MODEL = "anthropic/claude-haiku-4-5"
# MODEL = "ollama/llama3"

In [ ]:
from litellm import completion

response = completion(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is a Python tuple? Answer in one sentence."},
    ],
)

print(response.choices[0].message.content)
print(f"\nModel utilisé : {response.model}")
print(f"Tokens — prompt: {response.usage.prompt_tokens}, completion: {response.usage.completion_tokens}")